# Generation at n=200 — TPU edition

`solution_5_n200.ipynb` rewritten for a **Colab TPU**, with the defects listed in
`solution_5_n200_fixed.ipynb` fixed here too.

## Read this before running: the TPU forces a model change

The original runs **Qwen2.5-7B-Instruct in 4-bit** via bitsandbytes. Neither part survives the
move to TPU:

| | original (T4) | here (TPU) |
|---|---|---|
| quantisation | bitsandbytes 4-bit | **none** — bitsandbytes is CUDA-only, there is no TPU kernel |
| weights | ~4.5 GB (4-bit) | bf16 only |
| 7B in bf16 | — | ~15.2 GB, vs 16 GB HBM on a v5e-1 — will not fit once the KV cache is added |

So this notebook defaults to **Qwen2.5-3B-Instruct in bf16** (~6.2 GB), which is one of the
alternatives the original config already lists.

**Consequence: the numbers this produces are NOT comparable to the 7B/4-bit T4 run.** Different
model, different quantisation. Treat it as a separate experiment, not a continuation. Set
`llm` back to the 7B in CONFIG if you are on a TPU with enough HBM; the notebook checks and warns.

## What else changes for XLA

- **Static KV cache + fixed prompt length.** `generate()` on XLA recompiles for every new input
  shape, which would dominate the runtime. Prompts are padded to a fixed `prompt_len` and
  generation uses `cache_implementation="static"` with a fixed `max_new_tokens`, so each phase
  compiles once.
- **No `device_map="auto"`.** That is an accelerate/CUDA path; the model is loaded then moved to
  the XLA device explicitly.
- A **smoke test right after loading** fails fast if the XLA generate path is broken, rather than
  discovering it 600 generations in.

Falls back to CUDA then CPU automatically, and prints which backend it actually got.

### Install

In [2]:
import importlib.util, subprocess, sys

# Pin torch_xla to the ALREADY-INSTALLED torch so pip does not pull a different
# torch and force a runtime restart mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
         "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
        capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
else:
    print("torch_xla already available")

# NOTE: bitsandbytes is deliberately NOT installed -- it has no TPU backend.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "accelerate", "sentencepiece"], check=False)
print("deps ready")

torch_xla already available
deps ready


### Device — and which one we actually got

In [3]:
import os, gc, json, re, random, time
import numpy as np
import pandas as pd
import torch

BACKEND, device, xm = "cpu", torch.device("cpu"), None
try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"

def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        torch_xla.sync() if hasattr(torch_xla, "sync") else xm.mark_step()

print("=" * 80)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
else:
    print("NOT running on TPU.")
print("=" * 80)

BACKEND ACTUALLY IN USE: TPU   (device=xla:0)
torch 2.9.0+cpu | torch_xla 2.9.0


### Paths

The original hard-coded `corpus_v2.json` / `qa_pairs_wiki.json` in the working directory. This
resolver walks up for the repo's `data/` directory, then tries the working directory (what a Colab
upload gives you), then falls back to fetching from GitHub — and prints which source it used.
`corpus_v2.json` is committed in this repo as `data/corpus.json`.

In [4]:
import os, json, urllib.request
from pathlib import Path

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    """Nearest ancestor containing a data/ directory (works from notebooks/ or root)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def resolve(*names):
    """Local repo -> working dir -> GitHub. Returns (loaded_json, description)."""
    for n in names:
        if ROOT and (ROOT / "data" / n).exists():
            p = ROOT / "data" / n
            return json.loads(p.read_text(encoding="utf-8")), f"repo: {p}"
    for n in names:
        if Path(n).exists():
            return json.loads(Path(n).read_text(encoding="utf-8")), f"working dir: {n}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

# Outputs go to results/ when running inside the repo, else the working directory.
OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo -- will use working dir / GitHub)")
print("output dir:", OUT_DIR.resolve())

repo root : (not in the repo -- will use working dir / GitHub)
output dir: /content


### Config

In [5]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "finetuned_path": None,   # C3 dropped: no end-to-end effect at n=68, so no split is needed
    "alpha": 0.8,
    "top_k": 5,
    "n_eval": 200,

    # 3B, not 7B: see the memory table at the top. Qwen2.5-7B in bf16 is ~15.2 GB
    # and there is no 4-bit path on TPU, so it does not fit a 16 GB v5e-1.
    "llm": "Qwen/Qwen2.5-3B-Instruct",
    "llm_dtype": "bfloat16",       # native on TPU; fp32 would double the footprint

    # XLA needs static shapes. Prompts are padded to exactly prompt_len and
    # generation runs a fixed number of steps, so each phase compiles once.
    "prompt_len": 3072,
    "max_new_tokens": 128,
    "judge_max_new_tokens": 40,
    "batch_size": 8,

    "checkpoint": "gen_n200_tpu_checkpoint.json",
    "seed": 42,
}
CONFIG

{'base_encoder': 'intfloat/multilingual-e5-base',
 'finetuned_path': None,
 'alpha': 0.8,
 'top_k': 5,
 'n_eval': 200,
 'llm': 'Qwen/Qwen2.5-3B-Instruct',
 'llm_dtype': 'bfloat16',
 'prompt_len': 3072,
 'max_new_tokens': 128,
 'judge_max_new_tokens': 40,
 'batch_size': 8,
 'checkpoint': 'gen_n200_tpu_checkpoint.json',
 'seed': 42}

### Load data

In [6]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c)
print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# No train/test split needed: condition C3 (fine-tuned encoder) is dropped,
# since it showed no end-to-end effect at n=68 (C3 - C2 = +0.000, CI [-0.088, +0.074]).
# Seeded shuffle kept so the evaluation order is reproducible.
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | QA {len(qa)} | evaluating {len(eval_qa)} (full benchmark, no holdout)")

corpus from github: https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/corpus.json
qa     from github: https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/qa_pairs_wiki.json
Corpus 3054 | QA 200 | evaluating 200 (full benchmark, no holdout)


### BM25

In [7]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

BM25 index built


### Build contexts for all conditions, then free the encoder

In [8]:
from transformers import AutoTokenizer, AutoModel

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real); last batch padded up to bs to keep XLA shapes static."""
    for i in range(0, len(items), bs):
        ch = list(items[i:i + bs]); n = len(ch)
        if n < bs:
            ch += [ch[-1]] * (bs - n)
        yield ch, n

@torch.no_grad()
def encode_texts(model, tok, texts, bs=32, max_len=512, label=""):
    out, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(texts, bs):
        enc = tok(ch, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        v = mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        out.append(v.float().cpu().numpy()[:n]); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(out, 0).astype("float32")

enc_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
enc_model = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

corpus_emb = encode_texts(enc_model, enc_tok, [f"passage: {t}" for t in corpus_texts], label="corpus")
q_emb = {f: encode_texts(enc_model, enc_tok, [f"query: {q[f]}" for q in eval_qa], label=f)
         for f in ["msa_query", "darija_query"]}

K = CONFIG["top_k"]
def ids_for(field, i, q):
    s = CONFIG["alpha"] * minmax(corpus_emb @ q_emb[field][i]) + (1 - CONFIG["alpha"]) * minmax(bm25_scores(q[field]))
    return [corpus_ids[j] for j in np.argsort(-s)[:K]]

contexts = {
    "C1_msa_base":    {q["id"]: ids_for("msa_query", i, q) for i, q in enumerate(eval_qa)},
    "C2_darija_base": {q["id"]: ids_for("darija_query", i, q) for i, q in enumerate(eval_qa)},
    "C4_oracle":      {q["id"]: [q["source_chunk_id"]] for q in eval_qa},
}

del enc_model, corpus_emb, q_emb
gc.collect()

for cond, d in contexts.items():
    hit = sum(1 for q in eval_qa if q["source_chunk_id"] in d[q["id"]]) / len(eval_qa)
    print(f"  {cond:<22} gold in context: {hit:.1%}")

json.dump(contexts, open(out("contexts.json"), "w", encoding="utf-8"), ensure_ascii=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    corpus 640/3054 (5s)
    corpus 1280/3054 (5s)
    corpus 1920/3054 (6s)
    corpus 2560/3054 (7s)
  C1_msa_base            gold in context: 94.5%
  C2_darija_base         gold in context: 90.0%
  C4_oracle              gold in context: 100.0%


### Load the LLM

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"      # required for correct batched generation
tok.truncation_side = "left"   # overflow must drop CONTEXT, never the trailing instruction

DTYPE = getattr(torch, CONFIG["llm_dtype"])
llm = AutoModelForCausalLM.from_pretrained(CONFIG["llm"])   # no device_map: that is a CUDA path
llm = llm.to(device=device, dtype=DTYPE).eval()

n_params = sum(p.numel() for p in llm.parameters())
gb = n_params * DTYPE.itemsize / 1e9
print(f"Loaded {CONFIG['llm']} on {BACKEND.upper()}  |  {n_params/1e9:.2f}B params, ~{gb:.1f} GB in {CONFIG['llm_dtype']}")
if gb > 13:
    print("  WARNING: this is close to a 16 GB HBM budget once the KV cache is added.")
    print("  If generation OOMs, drop to Qwen/Qwen2.5-1.5B-Instruct.")

TRUNCATED = 0   # counted, not silent

@torch.no_grad()
def chat_batch(prompts, max_new_tokens):
    """Batched chat completion with STATIC shapes, so XLA compiles once per phase."""
    global TRUNCATED
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True)
             for p in prompts]
    for t in texts:
        if len(tok(t)["input_ids"]) > CONFIG["prompt_len"]:
            TRUNCATED += 1
    enc = tok(texts, return_tensors="pt", padding="max_length", truncation=True,
              max_length=CONFIG["prompt_len"])
    enc = {k: v.to(device) for k, v in enc.items()}
    # cache_implementation="static" preallocates the KV cache to a fixed size, so
    # shapes are already static and the graph compiles once. Do NOT also force
    # min_new_tokens: that would suppress EOS and make the model ramble past its
    # answer, appending junk to every generation and feeding stray yes/no tokens
    # to the judge parser. Early stopping just runs the same graph fewer times.
    out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                       do_sample=False, cache_implementation="static",
                       pad_token_id=tok.pad_token_id)
    sync()
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tok.decode(g, skip_special_tokens=True).strip() for g in gen]

# Fail fast: if the XLA generate path is broken, find out now, not 600 generations in.
t0 = time.time()
print("Smoke test:", chat_batch(["\u0623\u062c\u0628 \u0628\u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629: \u0645\u0627 \u0639\u0627\u0635\u0645\u0629 \u0627\u0644\u0645\u063a\u0631\u0628\u061f"], 20)[0])
print(f"(first call includes XLA compilation: {time.time()-t0:.0f}s)")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-3B-Instruct on TPU  |  3.40B params, ~6.8 GB in bfloat16
Smoke test: 
(first call includes XLA compilation: 23s)


### Prompts and judge parsing

`parse_judge` is the fixed version. The original walked the reply line by line with
`if "مدعوم" … elif "مطابق" …`, so both verdicts on a single line lost the correctness one.
It now searches each field independently and reports whether the reply parsed at all, so a
malformed judge reply is visible instead of silently scoring (0, 0).

In [10]:
GEN_PROMPT = """\u0623\u062c\u0628 \u0639\u0646 \u0627\u0644\u0633\u0624\u0627\u0644 \u0627\u0644\u062a\u0627\u0644\u064a \u0627\u0639\u062a\u0645\u0627\u062f\u0627 \u0641\u0642\u0637 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u0641\u0642\u0629.
\u0625\u0630\u0627 \u0644\u0645 \u062a\u0643\u0646 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0645\u0648\u062c\u0648\u062f\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635\u060c \u0642\u0644 \u0628\u0627\u0644\u0636\u0628\u0637: \u0627\u0644\u0645\u0639\u0644\u0648\u0645\u0629 \u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635
\u0644\u0627 \u062a\u0633\u062a\u0639\u0645\u0644 \u0623\u064a \u0645\u0639\u0631\u0641\u0629 \u062e\u0627\u0631\u062c\u064a\u0629. \u0623\u062c\u0628 \u0628\u062c\u0645\u0644\u0629 \u0648\u0627\u062d\u062f\u0629 \u0642\u0635\u064a\u0631\u0629 \u0641\u0642\u0637.

\u0627\u0644\u0646\u0635\u0648\u0635:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}

\u0627\u0644\u0625\u062c\u0627\u0628\u0629:"""

# Faithfulness and correctness are judged in ONE call to halve the work.
# They are separate constructs: an answer can faithfully report a passage that
# retrieval wrongly supplied, and so be faithful but incorrect.
JUDGE_PROMPT = """\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629: {gold}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629: {answer}

\u0623\u062c\u0628 \u0639\u0646 \u0633\u0624\u0627\u0644\u064a\u0646 \u0628\u062f\u0642\u0629:
1. \u0647\u0644 \u0643\u0644 \u0645\u0627 \u0648\u0631\u062f \u0641\u064a \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u062f\u0639\u0648\u0645 \u0635\u0631\u0627\u062d\u0629 \u0628\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629\u061f
2. \u0647\u0644 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u0637\u0627\u0628\u0642\u0629 \u0641\u064a \u0627\u0644\u0645\u0639\u0646\u0649 \u0644\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629\u061f \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0635\u064a\u0627\u063a\u0629 \u0645\u0642\u0628\u0648\u0644\u060c \u0623\u0645\u0627 \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0623\u0631\u0642\u0627\u0645 \u0623\u0648 \u0627\u0644\u0623\u0633\u0645\u0627\u0621 \u0623\u0648 \u0627\u0644\u062a\u0648\u0627\u0631\u064a\u062e \u0641\u063a\u064a\u0631 \u0645\u0642\u0628\u0648\u0644.

\u0623\u062c\u0628 \u0628\u0647\u0630\u0627 \u0627\u0644\u0634\u0643\u0644 \u0641\u0642\u0637 \u0648\u0628\u062f\u0648\u0646 \u0623\u064a \u0634\u0631\u062d:
\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627
\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627"""

REFUSAL = "\u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629"

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

_YES, _NO = "\u0646\u0639\u0645", "\u0644\u0627"
def _verdict(text, field):
    """Value of one labelled field, wherever it appears. None if absent or unusable.

    Searched per field rather than per line: the original's if/elif meant a reply
    with both verdicts on one line silently lost the second one. Whichever of
    yes/no appears first wins, so trailing commentary cannot flip the verdict,
    and a reply that merely echoes the template "\u0646\u0639\u0645/\u0644\u0627" counts as unparsed
    rather than as a spurious yes.
    """
    m = re.search(field + r"\s*[:\uFF1A]?\s*([^\n\u060c,]*)", text)
    if not m:
        return None
    v = m.group(1).strip()
    if re.fullmatch(_YES + r"\s*/\s*" + _NO, v):   # echoed the instruction verbatim
        return None
    iy, ino = v.find(_YES), v.find(_NO)
    if iy == -1 and ino == -1:
        return None
    if iy == -1:
        return 0
    if ino == -1:
        return 1
    return 1 if iy < ino else 0

def parse_judge(text):
    """-> (faithful, correct, parsed). `parsed` is 0 when the reply was unusable,
    which the original could not distinguish from a genuine double-negative."""
    t = (text or "").replace("\u060c", " ")
    f = _verdict(t, "\u0645\u062f\u0639\u0648\u0645")
    c = _verdict(t, "\u0645\u0637\u0627\u0628\u0642")
    parsed = int(f is not None and c is not None)
    return (f or 0), (c or 0), parsed

# Regression check for the bug that was fixed.
_c = [("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),   # normal two-line reply
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (0, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645 \u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),   # BOTH ON ONE LINE -- the original bug
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\u060c \u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),   # comma-separated
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627", (0, 0, 0)),  # echoed template -> unparsed
      ("", (0, 0, 0)),
      ("\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0635\u062d\u064a\u062d\u0629", (0, 0, 0))]
for txt, exp in _c:
    got = parse_judge(txt)
    assert got == exp, f"parse_judge regression: {txt!r} -> {got}, expected {exp}"
print("parse_judge: all regression cases pass (incl. both verdicts on one line)")

parse_judge: all regression cases pass (incl. both verdicts on one line)


### Run (batched + checkpointed)

In [11]:
from tqdm.auto import tqdm

CKPT = out(CONFIG["checkpoint"])
records = []
if os.path.exists(CKPT):
    records = json.load(open(CKPT, encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["condition"]) for r in records}

CONDITION_QUERY = {
    "C1_msa_base": "msa_query",
    "C2_darija_base": "darija_query",
    "C3_darija_finetuned": "darija_query",
    "C4_oracle": "darija_query",
}
byid = {q["id"]: q for q in eval_qa}
B = CONFIG["batch_size"]

for cond, ctx_map in contexts.items():
    qfield = CONDITION_QUERY[cond]
    todo = [q for q in eval_qa if (q["id"], cond) not in done]
    if not todo:
        continue
    print(f"\n=== {cond} ({len(todo)} to do) ===")

    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [ctx_map[q["id"]] for q in batch]

        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q[qfield])
                              for q, c in zip(batch, chunks)], CONFIG["max_new_tokens"])

        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c),
                                                   question=q["msa_query"],
                                                   gold=q["gold_answer"],
                                                   answer=a)
                               for q, c, a in zip(batch, chunks, answers)],
                              CONFIG["judge_max_new_tokens"])

        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok, parsed = parse_judge(v)
            records.append({
                "qid": q["id"], "condition": cond,
                "gold_in_context": int(q["source_chunk_id"] in c),
                "answer": a, "faithful": f, "correct": ok,
                "judge_parsed": parsed, "judge_raw": v,
                "refused": int(REFUSAL in (a or "")),
            })

        json.dump(records, open(CKPT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv(out("generation_local_raw.csv"), index=False)
print(f"\nComplete: {len(gen)} generations.")

# Both of these were silent in the original.
pf = 1 - gen.judge_parsed.mean()
print(f"Judge parse-failure rate: {pf:.1%}"
      + ("  <- metrics below are understated by this much" if pf > 0.01 else "  (negligible)"))
print(f"Prompts truncated: {TRUNCATED}"
      + ("  <- raise max prompt length" if TRUNCATED else "  (none)"))


=== C1_msa_base (200 to do) ===


  0%|          | 0/25 [00:00<?, ?it/s]

: 

### Results by condition

In [ ]:
print("=" * 88)
print("GENERATION RESULTS BY CONDITION")
print("=" * 88)
order = [c for c in CONDITION_QUERY if c in gen.condition.unique()]
summary = gen.groupby("condition").agg(
    n=("qid", "count"),
    gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"),
    correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"),
    judge_parsed=("judge_parsed", "mean"),
).reindex(order)

# Faithfulness among ANSWERED items only. A refusal is not "supported by the
# reference texts", so the judge scores refusals near-randomly -- in the committed
# n=68 run, 19 of 35 refusals were called faithful and 16 unfaithful. Reporting
# both columns keeps the headline number from being driven by refusal rate.
ans = gen[gen.refused == 0]
summary["faithfulness_answered"] = ans.groupby("condition")["faithful"].mean().reindex(order)
summary["correctness_answered"] = ans.groupby("condition")["correct"].mean().reindex(order)

print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv(out("generation_local_summary.csv"))

print("""
  gold_in_context        how often retrieval put the right passage before the generator
  faithfulness           answer supported by the context it was actually given
  correctness            answer matches the gold answer   <- what users care about
  refusal_rate           model declared the information absent
  judge_parsed           share of judge replies that parsed (1.000 = all good)
  *_answered             same metric excluding refusals -- see the note above
""")

### Paired comparisons with bootstrap CIs

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_cond, b_cond, col):
    a = gen[gen.condition == a_cond].set_index("qid")[col]
    b = gen[gen.condition == b_cond].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

print("=" * 88)
print("PAIRED COMPARISONS (95% CI)")
print("=" * 88)
rows = []
for a, b, label in [
    ("C1_msa_base", "C2_darija_base", "Does the dialect gap reach the answers?"),
    ("C3_darija_finetuned", "C2_darija_base", "Does the fine-tuned retriever help answers?"),
    ("C4_oracle", "C2_darija_base", "How much is lost to retrieval vs generation?"),
]:
    if a not in gen.condition.unique() or b not in gen.condition.unique():
        continue
    print(f"\n{label}   [{a} - {b}]")
    for col in ["correct", "faithful"]:
        d, lo, hi = paired(a, b, col)
        sig = "yes" if (lo > 0 or hi < 0) else "no"
        print(f"  {col:<10} {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  significant: {sig}")
        rows.append({"comparison": f"{a} - {b}", "metric": col,
                     "diff": d, "lo": lo, "hi": hi, "significant": sig})
pd.DataFrame(rows).to_csv(out("generation_local_comparisons.csv"), index=False)

### When retrieval misses, does the generator refuse or hallucinate?

The original printed a fixed paragraph asserting "low refusal + low correctness … the generator
does not notice the context is wrong and answers confidently anyway" — regardless of the data. In
the run it was printed under, the refusal rate on missing-gold rows was **0.737**, which is the
opposite behaviour. The conclusion is now read off the measured value.

In [ ]:
print("\n" + "=" * 88)
print("WHEN RETRIEVAL FAILS, WHAT DOES THE GENERATOR DO?")
print("=" * 88)
d2 = gen[gen.condition == "C2_darija_base"]
miss, hit = d2[d2.gold_in_context == 0], d2[d2.gold_in_context == 1]

if len(miss):
    for label, sub in [("MISSING", miss), ("PRESENT", hit)]:
        print(f"\nGold passage {label} ({len(sub)} cases):")
        print(f"  correctness   {sub.correct.mean():.3f}")
        print(f"  faithfulness  {sub.faithful.mean():.3f}")
        print(f"  refusal rate  {sub.refused.mean():.3f}")

    r = miss.refused.mean()
    print("\nInterpretation (derived from the numbers above, not assumed):")
    if r >= 0.5:
        print(f"  Refusal rate on missing-gold rows is {r:.3f} -- the generator mostly DETECTS")
        print("  that the retrieved context does not answer the question and declines, rather")
        print("  than inventing an answer. That is safe degradation: the end-to-end cost of a")
        print("  retrieval miss shows up as a non-answer, not as a confident falsehood.")
    elif r >= 0.2:
        print(f"  Refusal rate on missing-gold rows is {r:.3f} -- mixed. The generator catches")
        print("  some retrieval failures and answers through others.")
    else:
        print(f"  Refusal rate on missing-gold rows is {r:.3f} -- the failure the proposal")
        print("  predicted: the generator does not notice the context is wrong and answers")
        print("  confidently anyway.")
    print(f"\n  Correctness drops {hit.correct.mean():.3f} -> {miss.correct.mean():.3f} "
          f"when the gold passage is absent.")
else:
    print("Retrieval never missed in this sample - raise n_eval for this analysis.")

### Example failures for the paper

In [ ]:
print("\n=== Sample failures (dialect condition, incorrect answer) ===\n")
for _, r in gen[(gen.condition == "C2_darija_base") & (gen.correct == 0)].head(5).iterrows():
    q = byid[r["qid"]]
    print(f"Q (darija): {q['darija_query']}")
    print(f"Gold      : {q['gold_answer']}")
    print(f"Generated : {str(r['answer'])[:200]}")
    print(f"gold in context: {bool(r['gold_in_context'])} | faithful: {bool(r['faithful'])}"
          f" | refused: {bool(r['refused'])} | judge parsed: {bool(r['judge_parsed'])}")
    print("-" * 80)

print(f"\nAll outputs written to: {OUT_DIR.resolve()}")
for f in ["generation_local_raw.csv", "generation_local_summary.csv",
          "generation_local_comparisons.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab. Guarded so the notebook finishes cleanly anywhere.
try:
    from google.colab import files
    for f in ["generation_local_raw.csv", "generation_local_summary.csv",
              "generation_local_comparisons.csv"]:
        files.download(out(f))
except ImportError:
    print("\n(Not in Colab - files are on disk at the path above, no download needed.)")